# OpenCV Crash Course: 2026 Edition
## Module 7: Train YOLO on Your Own Objects
**Estimated time: 20 minutes**

- YOLO dataset format
- Train, check metrics, use `best.pt`
- Real dataset from Roboflow + free GPU on Colab

> Pretrained YOLO knows 80 classes. A helmet, a defect, a valve, a meter: **you train for it**. Most asked skill in CV jobs.

In [ ]:
# !pip install ultralytics

In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

def show(img, title="", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.show()

## 1. Dataset Format

```
my_dataset/
├── data.yaml          <- paths + class names
├── train/images/  train/labels/
└── valid/images/  valid/labels/
```

One `.txt` per image, one line per object (values 0..1):
```
class_id  x_center  y_center  width  height
0         0.512     0.430     0.210  0.380
```

Label your own images with **Roboflow** or **CVAT**. Start with ~100-300 images per class.

Below we use **coco8** (8 images, auto-download) so training runs on a laptop CPU.

In [ ]:
data = check_det_dataset('coco8.yaml')
root = data['path']
label = sorted(glob.glob(f"{root}/labels/train/*.txt"))[0]
print(open(label).read())

## 2. Train

| Arg | Real project value |
|---|---|
| `epochs` | 50-100 |
| `imgsz` | 640 |
| `batch` | 16 |
| `device` | `0` for GPU |

In [ ]:
model = YOLO('models/yolo26n.pt')                       # start from pretrained (transfer learning)
res = model.train(data='coco8.yaml', epochs=5, imgsz=320, batch=8,
                  name='coco8_demo', exist_ok=True, verbose=False)
best_path = f"{res.save_dir}/weights/best.pt"
print("Best model:", best_path)

## 3. Metrics

| Metric | Meaning | Good |
|---|---|---|
| Precision | predicted boxes that are correct | > 0.8 |
| Recall | real objects that were found | > 0.8 |
| mAP50 | overall accuracy | > 0.7 |

In [ ]:
best = YOLO(best_path)
m = best.val(data='coco8.yaml', imgsz=320, verbose=False)
print(f"Precision {m.box.mp:.2f} | Recall {m.box.mr:.2f} | mAP50 {m.box.map50:.2f}")

val_img = glob.glob(f"{root}/images/val/*.jpg")[0]
show(best(val_img, imgsz=320, verbose=False)[0].plot(), 'Prediction with best.pt')

## 4. Real Dataset on Google Colab (free GPU)

1. Colab: Runtime → Change runtime type → **T4 GPU**
2. universe.roboflow.com → pick a dataset → Download → format **YOLOv8** → copy the code

```python
!pip install roboflow ultralytics
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_KEY")
ds = rf.workspace("WORKSPACE").project("PROJECT").version(1).download("yolov8")

from ultralytics import YOLO
YOLO("yolo26n.pt").train(data=f"{ds.location}/data.yaml", epochs=50, imgsz=640, device=0)
```

3. Download `runs/detect/train/weights/best.pt` and use it like any model.

---
## Summary

| Task | Code |
|------|------|
| Train | `YOLO('models/yolo26n.pt').train(data='data.yaml', epochs=50)` |
| Validate | `model.val(data='data.yaml').box.map50` |
| Use | `YOLO('runs/detect/train/weights/best.pt')` |

**Next:** Module 8, Segmentation